# Pediatrician Scheduling Model

---
## 1. Sets and Indices

- Let $i = 1, 2, \ldots, 42$ be the shift number, where $i = 1$ is Week 1 Monday Day Shift, $i = 2$ is Week 1 Monday Night Shift, $\ldots$, $i = 42$ is Week 3 Sunday Night Shift
- Let $j = 1, 2, \ldots, 13$ represent the pediatrician
- Let $k = 1, 2, 3$ represent the week

**Shift numbering scheme:**

Each week contains 14 shifts (Day then Night for each of the 7 days). Weeks 2 and 3 continue sequentially (shifts 15–28 and 29–42, respectively).

| Week | Day | Day Shift $i$ | Night Shift $i$ |
|------|-----|:---:|:---:|
| 1 | Mon | 1 | 2 |
| 1 | Tue | 3 | 4 |
| 1 | Wed | 5 | 6 |
| 1 | Thu | 7 | 8 |
| 1 | Fri | 9 | 10 |
| 1 | Sat | 11 | 12 |
| 1 | Sun | 13 | 14 |
| 2 | Mon–Sun | 15, 17, 19, 21, 23, 25, 27 | 16, 18, 20, 22, 24, 26, 28 |
| 3 | Mon–Sun | 29, 31, 33, 35, 37, 39, 41 | 30, 32, 34, 36, 38, 40, 42 |

**Key subsets:**

- Let $D = \{1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 33, 35, 37, 39, 41\}$ denote the set of **day shifts** (odd-numbered)
- Let $N_k$ denote the set of night shifts in week $k$:
  - $N_1 = \{2, 4, 6, 8, 10, 12, 14\}$,  $N_2 = \{16, 18, 20, 22, 24, 26, 28\}$,  $N_3 = \{30, 32, 34, 36, 38, 40, 42\}$
- Let $W_k$ denote the set of **weekend shifts** (Friday Night, Saturday Day & Night, Sunday Day & Night) in week $k$:
  - $W_1 = \{10, 11, 12, 13, 14\}$,  $W_2 = \{24, 25, 26, 27, 28\}$,  $W_3 = \{38, 39, 40, 41, 42\}$
- Let $POW_k$ denote the set of shifts the **Pediatrician of the Week** must cover in week $k$ (Tuesday Day, Thursday Day, Saturday Night):
  - $POW_1 = \{3, 7, 12\}$,  $POW_2 = \{17, 21, 26\}$,  $POW_3 = \{31, 35, 40\}$
- Let $CN$ denote the set of **consecutive night shift pairs** $(i, i')$ where $i$ and $i'$ are night shifts on adjacent calendar days (including cross-week pairs $(14, 16)$ and $(28, 30)$)
- Let $T$ denote the set of **night shift pairs two calendar days apart** $(i, i')$: e.g., Monday Night & Wednesday Night within a week, or Saturday Night & Monday Night across weeks (including cross-week pairs $(12, 16)$ and $(26, 30)$)

## 2. Parameters

- Let $L_j$ be the workload (total number of shifts they should work) for this cycle for pediatrician $j$
- Let $A_{ij}$ be the availability of pediatrician $j$ to work shift $i$, where $A_{ij} = 1$ means available and $A_{ij} = 0$ means not available
- Let $P_j$ be the remaining times in the year that pediatrician $j$ is supposed to serve as the Pediatrician of the Week (POW). If $P_j = 0$, the ped cannot be assigned as POW this cycle
- Let $w_g$ be the weight assigned to goal $g = 1, 2, 3, 4$ in the objective function


## 3. Decision Variables

**Primary binary variables:**

- Let $x_{ij}$ be a binary variable indicating whether shift $i$ is covered by pediatrician $j$
- Let $y_{kj}$ be a binary variable indicating whether pediatrician $j$ is the POW for week $k$

**Auxiliary binary variable (for goal programming):**

- Let $z_{kj}$ be a binary variable indicating whether pediatrician $j$ works a weekend shift in week $k$

**Deviation variables — non-negative (for goal programming):**

| Variable | Gurobi name | Meaning |
|----------|-------------|----------|
| $cw_{kj} \geq 0$ | `cwDevVar[k,j]` | Captures whether ped $j$ works in **both** consecutive weekends $k$ and $k+1$ (for $k = 1, 2$) |
| $e2_{j,i,i'} \geq 0$ | `e2DevVar[j,i,ip]` | Captures violation of the "no night shifts two days apart" preference for ped $j$ and pair $(i, i') \in T$ |
| $e3_{kj} \geq 0$ | `e3DevVar[k,j]` | Captures excess night shifts above 2 for ped $j$ in week $k$ |
| $d_j^+ \geq 0$ | `dPlusVar[j]` | Over-deviation from equal day/night split for ped $j$ |
| $d_j^- \geq 0$ | `dMinusVar[j]` | Under-deviation from equal day/night split for ped $j$ |

In [1]:
#%pip install gurobipy
from gurobipy import Model, GRB
import pandas as pd
import gurobipy as gp

xl = pd.ExcelFile('PediatricianData.xlsx')

# ── Index sets ────────────────────────────────────────────────────────────────
ShiftNum = list(range(1, 43))   # i = 1 … 42
PedNum = list(range(1, 14))   # j = 1 … 13
WeekNum = list(range(1, 4))    # k = 1, 2, 3

#Number of Shifts Required to Work for each Pediatrician
ShiftReqData = xl.parse('ShiftsRequired')         # Pediatrician | ShiftsRequired
ShiftReqData = ShiftReqData.set_index('Pediatrician')

L = {j: ShiftReqData.loc[j, 'ShiftsRequired'] for j in PedNum}
 

#Shift availability  by each pediatrician (1 indicates pediatrician is available, 0 indicates not available)
AvailabilityData = pd.read_excel('PediatricianData.xlsx', sheet_name='ShiftRequests', 
                                 skiprows=2, nrows=43, usecols="D:Q",index_col=0)
A = {(int(i), int(j)): int(AvailabilityData.loc[i, j]) 
     for i in AvailabilityData.index 
     for j in AvailabilityData.columns}
 

#Number of remaining times that pediatrician is supposed to serve as the Pediatrician of the Week
POWReqData = pd.read_excel('PediatricianData.xlsx', sheet_name='RemainingPOWrequirements', index_col=0)
P = {j: int(POWReqData.loc[j, 'POWReqs']) for j in PedNum}

#  Subsets ───────────────────────────────────────────────────────────────────────
D = [i for i in ShiftNum if i % 2 == 1]  # day shift (odd numbers)

N_k = {                                     # night shifts per week
    1: [2,  4,  6,  8,  10, 12, 14],
    2: [16, 18, 20, 22, 24, 26, 28],
    3: [30, 32, 34, 36, 38, 40, 42],
}
W_k = {                                     # weekend shifts per week
    1: [10, 11, 12, 13, 14],               # Fri Night, Sat Day/Night, Sun Day/Night
    2: [24, 25, 26, 27, 28],
    3: [38, 39, 40, 41, 42],
}
POW_k = {                                   # required POW shifts per week
    1: [3,  7,  12],                        # Tue Day, Thu Day, Sat Night
    2: [17, 21, 26],
    3: [31, 35, 40],
}

# CN : consecutive night pairs (adjacent calendar-day nights, incl. cross-week)
CN = []
for w in WeekNum:
    nw = N_k[w]
    for idx in range(len(nw)-1) :
        CN.append((nw[idx], nw[idx+1]))
CN += [(14, 16), (28, 30)]    

# T : night pairs two calendar days apart (incl. cross-week)
T = []
for w in WeekNum:
    nw = N_k[w]
    for idx in range(len(nw)-2) :
        T.append((nw[idx], nw[idx+2]))
T += [(12,16), (26, 30)]

In [2]:
# =============================================================================
# MODEL
# =============================================================================

m = gp.Model("Pediatrician Scheduling")

# =============================================================================
# DECISION VARIABLES
# =============================================================================

# x[i, j] = 1  if shift i is covered by pediatrician j, 0 otherwise
xpedVar = m.addVars(ShiftNum,PedNum, vtype=GRB.BINARY, name="X")

#  y[k, j] = 1  if pediatrician j is the POW for week k, 0 otherwise
ypowVar = m.addVars(WeekNum,PedNum, vtype=GRB.BINARY, name="Y")
 
# z[k, j] = 1  if pediatrician j works a weekend shift in week k, 0 otherwise
weekendVar = m.addVars(WeekNum,PedNum, vtype=GRB.BINARY, name="Z")

# cw[k,j] >= 0 : G1 consecutive-weekend deviation (k = 1, 2 only)
cwDevVar   = m.addVars([1, 2],   PedNum,  vtype=GRB.CONTINUOUS, lb=0, name="CW")

# e2[j,i,ip] >= 0 : G2 nights-two-days-apart deviation
e2DevVar = m.addVars(PedNum, T, vtype=GRB.CONTINUOUS, lb=0, name="E2")
# e3[k,j] >= 0 : G3 excess-nights-per-week deviation
e3DevVar = m.addVars(WeekNum, PedNum, vtype=GRB.CONTINUOUS, lb=0, name="E3")

# d+[j], d-[j] >= 0 : G4 day/night split deviation
dPlusVar   = m.addVars(PedNum,            vtype=GRB.CONTINUOUS, lb=0, name="Dplus")
dMinusVar  = m.addVars(PedNum,            vtype=GRB.CONTINUOUS, lb=0, name="Dminus")

m.update()

Restricted license - for non-production use only - expires 2027-11-29


---
## 4. Hard Constraints

---

#### C1: Every Shift Must Have Exactly One Pediatrician

$$\sum_{j=1}^{13} x_{ij} = 1 \qquad \text{for all shifts } i = 1, \ldots, 42$$



#### C2: Every Pediatrician Must Work Their Expected Number of Shifts for the Cycle

$$\sum_{i=1}^{42} x_{ij} = L_j \qquad \text{for all pediatricians } j = 1, \ldots, 13$$



#### C3: Avoid Scheduling Anyone on Shifts They Requested Off

$$x_{ij} \leq A_{ij} \qquad \text{for all shifts } i \text{ and pediatricians } j$$



#### C4: No Back-to-Back Shifts (No 24 Consecutive Hours)

$$x_{ij} + x_{i+1,j} \leq 1 \qquad \forall\, i = 1, \ldots, 41;\; \forall\, j = 1, \ldots, 13$$


#### C5: No Consecutive Night Shifts

No pediatrician may work night shifts on two adjacent calendar days. $CN$ is the set of all such consecutive-night pairs (including cross-week pairs $(14,16)$ and $(28,30)$):

$$x_{ij} + x_{i'j} \leq 1 \qquad \forall\, (i, i') \in CN;\; \forall\, j = 1, \ldots, 13$$

#### C6: At Most 2 Shifts per Weekend

A weekend consists of Friday Night, Saturday Day & Night, and Sunday Day & Night ($|W_k| = 5$ shifts):

$$\sum_{i \in W_k} x_{ij} \leq 2 \qquad \forall\, k = 1, 2, 3;\; \forall\, j = 1, \ldots, 13$$

#### C7: Exactly One POW per Week

$$\sum_{j=1}^{13} y_{kj} = 1 \qquad \forall\, k = 1, 2, 3$$

#### C8: POW Must Cover Required Shifts (Tuesday Day, Thursday Day, Saturday Night)

$$x_{ij} \geq y_{kj} \qquad \forall\, i \in POW_k;\; \forall\, j = 1, \ldots, 13;\; \forall\, k = 1, 2, 3$$

#### C9: At Most One POW Assignment per Cycle

$$\sum_{k=1}^{3} y_{kj} \leq 1 \qquad \forall\, j = 1, \ldots, 13$$

#### C10: Peds with Fulfilled Annual POW Requirements Cannot be POW

Peds 3, 5, and 13 have $P_j = 0$:

$$\sum_{k=1}^{3} y_{kj} = 0 \qquad \forall\, j \text{ where } P_j = 0$$

#### C11: Binary Requirements

$$x_{ij} \in \{0, 1\} \qquad \forall\, i, j \qquad y_{kj} \in \{0, 1\} \qquad \forall\, k, j \qquad z_{kj} \in \{0, 1\} \qquad \forall\, k, j$$

In [3]:

# =============================================================================
# HARD CONSTRAINTS
# =============================================================================

#  C1: Every shift must have exactly one pediatrician ────────────────────────
# sum_j x[i,j] = 1   for all i
onePedCstr = m.addConstrs(
                (gp.quicksum(xpedVar[i,j] for j in PedNum) == 1 
                 for i in ShiftNum), 
                 name = 'onePed')
 
#  C2: Every ped must work their expected number of shifts ───────────────────
# sum_i x[i,j] = L_j   for all j
expectedShiftsCstr = m.addConstrs(
                        (gp.quicksum(xpedVar[i,j] for i in ShiftNum) == L[j]
                        for j in PedNum), 
                        name = 'expectedShifts')


# C3: Avoid scheduling anyone on shifts they requested off ──────────────────
# x[i,j] <= A[i,j]   for all i, j
availabilityCstr = m.addConstrs(
                        (xpedVar[i,j] <= A[i,j] 
                        for i in ShiftNum for j in PedNum), 
                        name = "availability") 

# C4: No back-to-back shifts (no 24 consecutive hours)
# x[i,j] + x[i+1,j] <= 1   for all i = 1..41, j
noBackToBackCstr = m.addConstrs(
                        (xpedVar[i,j] + xpedVar[i+1,j] <= 1
                         for i in ShiftNum[:-1] for j in PedNum),
                         name = "noBackToBack")

# C5: No consecutive night shifts (adjacent calendar-day nights)
# x[i,j] + x[i',j] <= 1   for all (i,i') in CN, j
noConsecutiveNightsCstr = m.addConstrs(
                                (xpedVar[i,j] + xpedVar[ip,j] <= 1
                                 for (i, ip) in CN for j in PedNum),
                                 name = "noConsecutiveNights")

# C6: At most 2 shifts per weekend
# sum_{i in W_k} x[i,j] <= 2   for all k, j
weekendCapCstr = m.addConstrs(
                        (gp.quicksum(xpedVar[i,j] for i in W_k[k]) <= 2
                         for k in WeekNum for j in PedNum),
                         name = "weekendCap")

# C7: Exactly one POW per week
# sum_j y[k,j] = 1   for all k
onePOWCstr = m.addConstrs(
                        (gp.quicksum(ypowVar[k,j] for j in PedNum) == 1
                         for k in WeekNum),
                         name = "onePOW")

# C8: POW must cover required shifts (Tue Day, Thu Day, Sat Night)
# x[i,j] >= y[k,j]   for all i in POW_k, j, k
powShiftCstr = m.addConstrs(
                        (xpedVar[i,j] >= ypowVar[k,j] 
                         for k in WeekNum for j in PedNum for i in POW_k[k]),
                         name = "powShiftCstr")

# C9: At most one POW assignment per cycle
# sum_k y[k,j] <= 1   for all j
onePOWperCycleCstr = m.addConstrs(
    (gp.quicksum(ypowVar[k, j] for k in WeekNum) <= 1
     for j in PedNum),
    name='onePOWperCycle'
)

# C10: Peds with P[j] = 0 cannot be POW this cycle
# sum_k y[k,j] = 0   for all j where P[j] = 0
noPOWCstr = m.addConstrs(
    (gp.quicksum(ypowVar[k, j] for k in WeekNum) == 0
     for j in PedNum if P[j] == 0),
    name='POWfulfilled'
)


---
## 5. Soft Constraints and Goal Programming

The following constraints reflect the *preferences* of the pediatricians. Violating them does not breach hospital policy, but doing so reduces schedule quality. Because there are **four competing preferences**, we apply **goal programming**: deviation variables measure how far the schedule falls from each goal, then a weighted sum of deviations is minimized.

---

### G1: Prefer Not to Work Consecutive Weekends

First, link the auxiliary variable $z_{kj}$ to whether ped $j$ worked any weekend shift in week $k$:

$$z_{kj} \leq \sum_{i \in W_k} x_{ij} \qquad \forall\, k, j \tag{G1a}$$

$$\sum_{i \in W_k} x_{ij} \leq 5 \cdot z_{kj} \qquad \forall\, k, j \quad (|W_k| = 5) \tag{G1b}$$

A violation occurs when ped $j$ works in **both** consecutive weekends $k$ and $k+1$. The deviation variable $cw_{kj}$ captures this:

$$cw_{kj} \geq z_{kj} + z_{k+1,j} - 1 \qquad \forall\, k = 1, 2;\; \forall\, j = 1, \ldots, 13 \tag{G1c}$$

> $cw_{kj} > 0$ means ped $j$ worked both weekends $k$ and $k+1$.

---

### G2: Prefer Not to Work Night Shifts Two Days Apart

For each pair $(i, i') \in T$ and each ped $j$, the soft goal is $x_{ij} + x_{i'j} \leq 1$. The deviation variable $e2_{j,i,i'}$ absorbs any violation:

$$x_{ij} + x_{i'j} \leq 1 + e2_{j,i,i'} \qquad \forall\, (i, i') \in T;\; \forall\, j = 1, \ldots, 13$$

---

### G3: Prefer No More Than Two Night Shifts per Week

The ideal is $\leq 2$ night shifts per ped per week. The deviation variable $e3_{kj}$ captures the excess:

$$\sum_{i \in N_k} x_{ij} \leq 2 + e3_{kj} \qquad \forall\, k = 1, 2, 3;\; \forall\, j = 1, \ldots, 13$$

---

### G4: Equitable Day/Night Split

The target day shifts for ped $j$ is $\lfloor L_j / 2 \rfloor$. Deviation variables $d_j^+$ and $d_j^-$ track over- and under-assignment of day shifts:

$$\sum_{i \in D} x_{ij} + d_j^- - d_j^+ = \left\lfloor \frac{L_j}{2} \right\rfloor \qquad \forall\, j = 1, \ldots, 13$$

> $d_j^+ > 0$ means more day shifts than target; $d_j^- > 0$ means fewer day shifts than target.

In [4]:
# =============================================================================
# SOFT CONSTRAINTS (Goal Programming)
# =============================================================================

# G1a: z[k,j] <= sum_{i in W_k} x[i,j]   (lower link for weekend indicator)
g1aLowerCstr = m.addConstrs(
    (weekendVar[k, j] <= gp.quicksum(xpedVar[i, j] for i in W_k[k])
     for k in WeekNum for j in PedNum),
    name='weekendLower'
)

# G1b: sum_{i in W_k} x[i,j] <= 5 * z[k,j]   (upper link, |W_k| = 5)
g1bUpperCstr = m.addConstrs(
    (gp.quicksum(xpedVar[i, j] for i in W_k[k]) <= 5 * weekendVar[k, j]
     for k in WeekNum for j in PedNum),
    name='weekendUpper'
)

# G1c: cw[k,j] >= z[k,j] + z[k+1,j] - 1   (consecutive-weekend deviation)
consecWeekendPref = m.addConstrs(
    (cwDevVar[k, j] >= weekendVar[k, j] + weekendVar[k + 1, j] - 1
     for k in [1, 2] for j in PedNum),
    name='consecWeekendPref'
)

# G2: x[i,j] + x[i',j] <= 1 + e2[j,i,i']   (nights two days apart)
# Note: e2DevVar key is (j, i, ip)
twoApartNightsPref = m.addConstrs(
    (xpedVar[i, j] + xpedVar[ip, j] <= 1 + e2DevVar[j, i, ip]
     for (i, ip) in T for j in PedNum),
    name='twoApartNights'
)

# G3: sum_{i in N_k} x[i,j] <= 2 + e3[k,j]   (max 2 night shifts per week)
nightExcessPref = m.addConstrs(
    (gp.quicksum(xpedVar[i, j] for i in N_k[k]) <= 2 + e3DevVar[k, j]
     for k in WeekNum for j in PedNum),
    name='nightExcess'
)

# G4: sum_{i in D} x[i,j] + d-[j] - d+[j] = floor(L[j]/2)   (day/night split)
dayNightSplitPref = m.addConstrs(
    (gp.quicksum(xpedVar[i, j] for i in D) + dMinusVar[j] - dPlusVar[j] == L[j] // 2
     for j in PedNum),
    name='dayNightSplit'
)


---
## 6. Objective Function

**Minimise** a weighted sum of all goal deviations:

$$\min \; Z = w_1 \sum_{k=1}^{2} \sum_{j=1}^{13} cw_{kj} \;+\; w_2 \sum_{j=1}^{13} \sum_{(i,i') \in T} e2_{j,i,i'} \;+\; w_3 \sum_{k=1}^{3} \sum_{j=1}^{13} e3_{kj} \;+\; w_4 \sum_{j=1}^{13} \left( d_j^+ + d_j^- \right)$$

| Term | Deviation variable | Goal |
|------|--------------------|------|
| $w_1$ | $cw_{kj}$ | G1: No consecutive weekends |
| $w_2$ | $e2_{j,i,i'}$ | G2: No nights two days apart  |
| $w_3$ | $e3_{kj}$ | G3: ≤ 2 night shifts per week |
| $w_4$ | $d_j^+, d_j^-$ | G4: Equitable day/night split |

> Adjust $w_1, w_2, w_3, w_4$ to reflect Meg's priorities. Equal weights (all = 1) treat all four goals as equally important.

In [5]:
# =============================================================================
# OBJECTIVE FUNCTION
# =============================================================================
w1, w2, w3, w4 = 1, 1, 1, 1    # equal weights — adjust to reflect priorities

obj = (
      w1 * gp.quicksum(cwDevVar[k, j]              for k in [1, 2]  for j in PedNum)
    + w2 * gp.quicksum(e2DevVar[j, i, ip]          for j in PedNum  for (i, ip) in T)
    + w3 * gp.quicksum(e3DevVar[k, j]              for k in WeekNum for j in PedNum)
    + w4 * gp.quicksum(dPlusVar[j] + dMinusVar[j]  for j in PedNum)
)

m.setObjective(obj, GRB.MINIMIZE)
m.update()

print(f'Variables  : {m.NumVars}')
print(f'Constraints: {m.NumConstrs}')
print('Model ready — call m.optimize() to solve.')


Variables  : 936
Constraints: 1946
Model ready — call m.optimize() to solve.


---
## 7. Solve and Results

##### Note: AI has been used to generate the result script to make it easier to interpret

In [8]:

# =============================================================================
# SOLVE & RESULTS
# =============================================================================
m.write('Pediatrician Scheduling.lp')
m.optimize()

if m.Status == GRB.OPTIMAL:
    print(f'\nOptimal solution found.  Objective Z = {m.ObjVal:.4f}')

    # ── Schedule: which ped covers each shift ─────────────────────────────────
    days_list  = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    shift_type = ['Day', 'Night']

    print('\n' + '='*55)
    print('SCHEDULE')
    print('='*55)
    print(f'{"Shift":>6}  {"Week":>4}  {"Day":>5}  {"Type":>6}  {"Ped":>4}')
    print('-'*55)
    for i in ShiftNum:
        week  = (i - 1) // 14 + 1
        day   = days_list[((i - 1) % 14) // 2]
        stype = shift_type[(i - 1) % 2]
        ped   = next(j for j in PedNum if xpedVar[i, j].X > 0.5)
        print(f'{i:>6}  {week:>4}  {day:>5}  {stype:>6}  {ped:>4}')

    # ── Workload check: assigned vs required ──────────────────────────────────
    print('\n' + '='*55)
    print('WORKLOAD CHECK (assigned vs required)')
    print('='*55)
    print(f'  {"Ped":>4}  {"Required":>9}  {"Assigned":>9}  {"Shifts"}')
    print('-'*55)
    for j in PedNum:
        assigned_shifts = [i for i in ShiftNum if xpedVar[i, j].X > 0.5]
        print(f'  {j:>4}  {L[j]:>9}  {len(assigned_shifts):>9}  {assigned_shifts}')

    # ── POW assignments ───────────────────────────────────────────────────────
    print('\n' + '='*55)
    print('POW ASSIGNMENTS')
    print('='*55)
    for k in WeekNum:
        pow_ped = next(j for j in PedNum if ypowVar[k, j].X > 0.5)
        req     = POW_k[k]
        covered = [i for i in req if xpedVar[i, pow_ped].X > 0.5]
        print(f'  Week {k}: Ped {pow_ped}  '
              f'(must cover shifts {req} → assigned {covered})')

    # ── Weekend indicator z[k,j] ──────────────────────────────────────────────
    print('\n' + '='*55)
    print('WEEKEND INDICATOR z[k,j]  (1 = worked that weekend)')
    print('='*55)
    print(f'  {"Ped":>4}  {"Wk1":>5}  {"Wk2":>5}  {"Wk3":>5}  {"cw[1,j]":>8}  {"cw[2,j]":>8}')
    print('-'*55)
    for j in PedNum:
        z1 = int(weekendVar[1, j].X > 0.5)
        z2 = int(weekendVar[2, j].X > 0.5)
        z3 = int(weekendVar[3, j].X > 0.5)
        cw1 = cwDevVar[1, j].X
        cw2 = cwDevVar[2, j].X
        flag1 = ' ← G1 violation' if cw1 > 0.001 else ''
        flag2 = ' ← G1 violation' if cw2 > 0.001 else ''
        if z1 or z2 or z3:
            print(f'  {j:>4}  {z1:>5}  {z2:>5}  {z3:>5}  '
                  f'{cw1:>8.3f}{flag1}  {cw2:>8.3f}{flag2}')

    # ── G2: nights two days apart violations ──────────────────────────────────
    print('\n' + '='*55)
    print('G2: NIGHTS TWO DAYS APART VIOLATIONS')
    print('='*55)
    g2_violations = []
    for j in PedNum:
        for (i, ip) in T:
            val = e2DevVar[j, i, ip].X
            if val > 0.001:
                g2_violations.append((j, i, ip, val))
                print(f'  Ped {j}: worked night shifts {i} and {ip} '
                      f'(2 days apart)  e2={val:.3f}')
    if not g2_violations:
        print('  None — G2 fully satisfied ✓')

    # ── G3: night excess per week ─────────────────────────────────────────────
    print('\n' + '='*55)
    print('G3: NIGHT EXCESS PER WEEK (target ≤ 2)')
    print('='*55)
    g3_violations = []
    for k in WeekNum:
        for j in PedNum:
            val = e3DevVar[k, j].X
            nights = [i for i in N_k[k] if xpedVar[i, j].X > 0.5]
            if val > 0.001:
                g3_violations.append((k, j, val))
                print(f'  Ped {j} Week {k}: {len(nights)} night shifts {nights}  '
                      f'e3={val:.3f} (exceeded target by {val:.0f})')
    if not g3_violations:
        print('  None — G3 fully satisfied ✓')

    # ── G4: day/night split ───────────────────────────────────────────────────
    print('\n' + '='*55)
    print('G4: DAY/NIGHT SPLIT (target = floor(L[j]/2) day shifts)')
    print('='*55)
    print(f'  {"Ped":>4}  {"L[j]":>5}  {"Target":>7}  {"Days":>5}  '
          f'{"Nights":>7}  {"d+":>5}  {"d-":>5}')
    print('-'*55)
    for j in PedNum:
        target = L[j] // 2
        day_shifts   = [i for i in D          if xpedVar[i, j].X > 0.5]
        night_shifts = [i for i in ShiftNum
                        if i not in D and xpedVar[i, j].X > 0.5]
        dp = dPlusVar[j].X
        dm = dMinusVar[j].X
        flag = '  ← G4 deviation' if dp > 0.001 or dm > 0.001 else ''
        print(f'  {j:>4}  {L[j]:>5}  {target:>7}  {len(day_shifts):>5}  '
              f'{len(night_shifts):>7}  {dp:>5.1f}  {dm:>5.1f}{flag}')

    # ── Objective breakdown ───────────────────────────────────────────────────
    print('\n' + '='*55)
    print('OBJECTIVE BREAKDOWN  (all weights = 1)')
    print('='*55)
    cw_sum = sum(cwDevVar[k, j].X for k in [1,2] for j in PedNum)
    e2_sum = sum(e2DevVar[j, i, ip].X for j in PedNum for (i, ip) in T)
    e3_sum = sum(e3DevVar[k, j].X for k in WeekNum for j in PedNum)
    dp_sum = sum(dPlusVar[j].X  for j in PedNum)
    dm_sum = sum(dMinusVar[j].X for j in PedNum)
    print(f'  G1 consecutive weekends  w1 * Σcw  = 1 × {cw_sum:.3f} = {w1*cw_sum:.3f}')
    print(f'  G2 nights two days apart w2 * Σe2  = 1 × {e2_sum:.3f} = {w2*e2_sum:.3f}')
    print(f'  G3 night excess per week w3 * Σe3  = 1 × {e3_sum:.3f} = {w3*e3_sum:.3f}')
    print(f'  G4 day/night imbalance   w4 * Σd   = 1 × {dp_sum+dm_sum:.3f} = {w4*(dp_sum+dm_sum):.3f}')
    print(f'  {"─"*45}')
    print(f'  Total Z = {m.ObjVal:.4f}')

else:
    status_codes = {2:'Infeasible', 3:'Unbounded', 4:'Infeasible or Unbounded',
                    5:'Numerical Issues', 9:'Time Limit'}
    print(f'\nNo optimal solution. Status: {status_codes.get(m.Status, m.Status)}')

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.3.0 25D125)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1946 rows, 936 columns and 5560 nonzeros (Min)
Model fingerprint: 0x4a7cac4a
Model has 312 linear objective coefficients
Variable types: 312 continuous, 624 integer (624 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolved: 498 rows, 608 columns, 2754 nonzeros

Continuing optimization...


Explored 1 nodes (231 simplex iterations) in 0.17 seconds (0.03 work units)
Most recent optimization runtime was 0.10 seconds (0.00 work units)
Thread count was 8 (of 8 available processors)

Solution count 3: 4 16 18 

Optimal solution found (tolerance 1.00e-04)
Best objective 4.000000000000e+00, best bound 4.000000000000e+00, gap 0.0000%

Optimal solution found.  Object

---
## 8. Interpretation of Solution

### 8.1 Feasibility of the Schedule

The model found an **optimal feasible solution** in 0.06 seconds, confirming that a valid 3-week schedule exists under all hospital policies. Every hard constraint was satisfied:

| Constraint | Result |
|-----------|--------|
| C1: One ped per shift | ✓ All 42 shifts covered by exactly one ped |
| C2: Required workloads | ✓ All 13 peds assigned exactly their required $L_j$ shifts |
| C3: Shift requests | ✓ No ped was scheduled for a shift they requested off |
| C4: No back-to-back shifts | ✓ No ped works two consecutive shifts (24 hrs in a row) |
| C5: No consecutive nights | ✓ No ped works night shifts on adjacent calendar days |
| C6: Weekend cap | ✓ No ped works more than 2 shifts in any single weekend |
| C7: One POW per week | ✓ Exactly one POW designated for each of the 3 weeks |
| C8: POW required shifts | ✓ Each POW covers their mandatory Tue Day, Thu Day, Sat Night |
| C9: One POW per cycle | ✓ No ped serves as POW more than once in the cycle |
| C10: Fulfilled POW quotas | ✓ Peds 3, 5, and 13 ($P_j = 0$) were not assigned as POW |

The **POW assignments** were: Ped 10 (Week 1), Ped 8 (Week 2), Ped 11 (Week 3).

---

### 8.2 Preference Satisfaction and Implications

The optimal objective value is $Z = 4$, meaning the schedule deviates from the four soft preferences by a total of 4 units. Here is how each goal performed:

| Goal | Result | Implications |
|------|:------:|--------------|
| G1: Consecutive Weekends | ✓ **Fully Satisfied** | This was the most concerning preference since working two or more consecutive weekends is highly disruptive to personal life and rest. The schedule completely avoids this. |
| G2: Night Shifts Two Days Apart | ✓ **Fully Satisfied** | No ped faces a fragmented sleep schedule from near-consecutive nights. This is important for maintaining alertness in a clinical environment. |
| G3: Night Shifts Per Week | ✓ **Fully Satisfied** | No ped exceeds 2 night shifts in any single week, which helps avoid weekly burnout and keeps clinical performance consistent. |
| G4: Day/Night Split | ⚠ **Minor Violations** | Four peds (1, 7, 8, 10) have one more day shift than their target $\lfloor L_j / 2 \rfloor$. This is mathematically unavoidable when $L_j$ is odd, and is limited to one extra shift each. |

---
## 9. Weighted Goal Programming

In the equal-weight run ($w_1 = w_2 = w_3 = w_4 = 1$), the solver treated all four preferences as equally important. In practice, however, some preferences have stronger clinical and personal implications than others. We assign the following weights according to the importance of each goal:

$$w_1 = 4, \quad w_2 = 4, \quad w_3 = 1, \quad w_4 = 1$$

The new objective is:

$$\min \; Z = 4 \sum_{k=1}^{2}\sum_{j} cw_{kj} + 4 \sum_{j}\sum_{(i,i') \in T} e2_{j,i,i'} + 1 \sum_{k=1}^{3}\sum_{j} e3_{kj} + 1 \sum_{j}\left(d_j^+ + d_j^-\right)$$

In [11]:
# =============================================================================
# WEIGHTED GOAL PROGRAMMING — new weights w1=4, w2=4, w3=1, w4=1
# =============================================================================

m2 = gp.Model('PedScheduling_Weighted')
m2.Params.OutputFlag = 1   # show solver log

# ── Decision variables─────────────────────────────
xpedVar2    = m2.addVars(ShiftNum, PedNum,  vtype=GRB.BINARY,     name='X')
ypowVar2    = m2.addVars(WeekNum,  PedNum,  vtype=GRB.BINARY,     name='Y')
weekendVar2 = m2.addVars(WeekNum,  PedNum,  vtype=GRB.BINARY,     name='Z')
cwDevVar2   = m2.addVars([1, 2],   PedNum,  vtype=GRB.CONTINUOUS, lb=0, name='CW')
e2DevVar2   = m2.addVars(PedNum,   T,       vtype=GRB.CONTINUOUS, lb=0, name='E2')
e3DevVar2   = m2.addVars(WeekNum,  PedNum,  vtype=GRB.CONTINUOUS, lb=0, name='E3')
dPlusVar2   = m2.addVars(PedNum,            vtype=GRB.CONTINUOUS, lb=0, name='Dplus')
dMinusVar2  = m2.addVars(PedNum,            vtype=GRB.CONTINUOUS, lb=0, name='Dminus')

# ── Hard constraints ────────────────────────────
m2.addConstrs((gp.quicksum(xpedVar2[i,j] for j in PedNum)==1 for i in ShiftNum), name='onePed')
m2.addConstrs((gp.quicksum(xpedVar2[i,j] for i in ShiftNum)==L[j] for j in PedNum), name='expectedShifts')
m2.addConstrs((xpedVar2[i,j]<=A[i,j] for i in ShiftNum for j in PedNum), name='availability')
m2.addConstrs((xpedVar2[i,j]+xpedVar2[i+1,j]<=1 for i in ShiftNum[:-1] for j in PedNum), name='noBackToBack')
m2.addConstrs((xpedVar2[i,j]+xpedVar2[ip,j]<=1 for (i,ip) in CN for j in PedNum), name='noConsecNights')
m2.addConstrs((gp.quicksum(xpedVar2[i,j] for i in W_k[k])<=2 for k in WeekNum for j in PedNum), name='weekendCap')
m2.addConstrs((gp.quicksum(ypowVar2[k,j] for j in PedNum)==1 for k in WeekNum), name='onePOW')
m2.addConstrs((xpedVar2[i,j]>=ypowVar2[k,j] for k in WeekNum for j in PedNum for i in POW_k[k]), name='powShifts')
m2.addConstrs((gp.quicksum(ypowVar2[k,j] for k in WeekNum)<=1 for j in PedNum), name='onePOWperCycle')
m2.addConstrs((gp.quicksum(ypowVar2[k,j] for k in WeekNum)==0 for j in PedNum if P[j]==0), name='POWfulfilled')

# ── Soft constraints ────────────────────────────
m2.addConstrs((weekendVar2[k,j]<=gp.quicksum(xpedVar2[i,j] for i in W_k[k]) for k in WeekNum for j in PedNum), name='weekendLower')
m2.addConstrs((gp.quicksum(xpedVar2[i,j] for i in W_k[k])<=5*weekendVar2[k,j] for k in WeekNum for j in PedNum), name='weekendUpper')
m2.addConstrs((cwDevVar2[k,j]>=weekendVar2[k,j]+weekendVar2[k+1,j]-1 for k in [1,2] for j in PedNum), name='consecWeekendPref')
m2.addConstrs((xpedVar2[i,j]+xpedVar2[ip,j]<=1+e2DevVar2[j,i,ip] for (i,ip) in T for j in PedNum), name='twoApartNights')
m2.addConstrs((gp.quicksum(xpedVar2[i,j] for i in N_k[k])<=2+e3DevVar2[k,j] for k in WeekNum for j in PedNum), name='nightExcess')
m2.addConstrs((gp.quicksum(xpedVar2[i,j] for i in D)+dMinusVar2[j]-dPlusVar2[j]==L[j]//2 for j in PedNum), name='dayNightSplit')

# ── Weighted objective — w1=4, w2=4, w3=1, w4=1 ──────────────────────────────
w1, w2, w3, w4 = 4, 4, 1, 1

obj2 = (
      w1 * gp.quicksum(cwDevVar2[k,j]              for k in [1,2]  for j in PedNum)
    + w2 * gp.quicksum(e2DevVar2[j,i,ip]           for j in PedNum for (i,ip) in T)
    + w3 * gp.quicksum(e3DevVar2[k,j]              for k in WeekNum for j in PedNum)
    + w4 * gp.quicksum(dPlusVar2[j]+dMinusVar2[j]  for j in PedNum)
)

m2.setObjective(obj2, GRB.MINIMIZE)
m2.update()
m2.optimize()

# =============================================================================
# WEIGHTED MODEL — RESULTS
# =============================================================================
if m2.Status == GRB.OPTIMAL:
    print(f'Optimal solution found.  Weighted Z = {m2.ObjVal:.4f}')

    days_list  = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    shift_type = ['Day', 'Night']

    # ── Schedule ──────────────────────────────────────────────────────────────
    print('\n' + '='*55)
    print('SCHEDULE')
    print('='*55)
    print(f'{"Shift":>6}  {"Week":>4}  {"Day":>5}  {"Type":>6}  {"Ped":>4}')
    print('-'*55)
    for i in ShiftNum:
        week  = (i-1)//14+1
        day   = days_list[((i-1)%14)//2]
        stype = shift_type[(i-1)%2]
        ped   = next(j for j in PedNum if xpedVar2[i,j].X > 0.5)
        print(f'{i:>6}  {week:>4}  {day:>5}  {stype:>6}  {ped:>4}')

    # ── POW assignments ───────────────────────────────────────────────────────
    print('\n' + '='*55)
    print('POW ASSIGNMENTS')
    print('='*55)
    for k in WeekNum:
        pow_ped = next(j for j in PedNum if ypowVar2[k,j].X > 0.5)
        req     = POW_k[k]
        covered = [i for i in req if xpedVar2[i,pow_ped].X > 0.5]
        print(f'  Week {k}: Ped {pow_ped}  (must cover {req} → assigned {covered})')

    # ── G1: Weekend indicators ────────────────────────────────────────────────
    print('\n' + '='*55)
    print('G1: WEEKEND INDICATOR z[k,j] AND DEVIATION cw[k,j]')
    print('='*55)
    print(f'  {"Ped":>4}  {"Wk1":>5}  {"Wk2":>5}  {"Wk3":>5}  {"cw[1,j]":>8}  {"cw[2,j]":>8}')
    print('-'*55)
    for j in PedNum:
        z1 = int(weekendVar2[1,j].X > 0.5)
        z2 = int(weekendVar2[2,j].X > 0.5)
        z3 = int(weekendVar2[3,j].X > 0.5)
        cw1 = cwDevVar2[1,j].X
        cw2 = cwDevVar2[2,j].X
        flag = '  ← G1 VIOLATION' if cw1>0.001 or cw2>0.001 else ''
        if z1 or z2 or z3:
            print(f'  {j:>4}  {z1:>5}  {z2:>5}  {z3:>5}  {cw1:>8.3f}  {cw2:>8.3f}{flag}')

    # ── G2 ────────────────────────────────────────────────────────────────────
    print('\n' + '='*55)
    print('G2: NIGHTS TWO DAYS APART VIOLATIONS')
    print('='*55)
    g2_found = False
    for j in PedNum:
        for (i,ip) in T:
            val = e2DevVar2[j,i,ip].X
            if val > 0.001:
                print(f'  Ped {j}: nights {i} & {ip}  e2={val:.3f}')
                g2_found = True
    if not g2_found:
        print('  None — G2 fully satisfied ✓')

    # ── G3 ────────────────────────────────────────────────────────────────────
    print('\n' + '='*55)
    print('G3: NIGHT EXCESS PER WEEK (target ≤ 2)')
    print('='*55)
    g3_found = False
    for k in WeekNum:
        for j in PedNum:
            val = e3DevVar2[k,j].X
            if val > 0.001:
                nights = [i for i in N_k[k] if xpedVar2[i,j].X > 0.5]
                print(f'  Ped {j} Week {k}: {len(nights)} nights {nights}  e3={val:.3f}')
                g3_found = True
    if not g3_found:
        print('  None — G3 fully satisfied ✓')

    # ── G4 ────────────────────────────────────────────────────────────────────
    print('\n' + '='*55)
    print('G4: DAY/NIGHT SPLIT')
    print('='*55)
    print(f'  {"Ped":>4}  {"L[j]":>5}  {"Target":>7}  {"Days":>5}  {"Nights":>7}  {"d+":>5}  {"d-":>5}')
    print('-'*55)
    for j in PedNum:
        target       = L[j]//2
        day_cnt      = sum(1 for i in D if xpedVar2[i,j].X > 0.5)
        night_cnt    = sum(1 for i in ShiftNum if i not in D and xpedVar2[i,j].X > 0.5)
        dp = dPlusVar2[j].X
        dm = dMinusVar2[j].X
        flag = '  ← G4 deviation' if dp>0.001 or dm>0.001 else ''
        print(f'  {j:>4}  {L[j]:>5}  {target:>7}  {day_cnt:>5}  {night_cnt:>7}  {dp:>5.1f}  {dm:>5.1f}{flag}')

    # ── Objective breakdown ───────────────────────────────────────────────────
    cw_sum = sum(cwDevVar2[k,j].X for k in [1,2] for j in PedNum)
    e2_sum = sum(e2DevVar2[j,i,ip].X for j in PedNum for (i,ip) in T)
    e3_sum = sum(e3DevVar2[k,j].X for k in WeekNum for j in PedNum)
    dp_sum = sum(dPlusVar2[j].X for j in PedNum)
    dm_sum = sum(dMinusVar2[j].X for j in PedNum)

    print('\n' + '='*55)
    print('OBJECTIVE BREAKDOWN  (weighted)')
    print('='*55)
    print(f'  G1: w1 * Σcw  = 4 × {cw_sum:.3f} = {4*cw_sum:.3f}')
    print(f'  G2: w2 * Σe2  = 3 × {e2_sum:.3f} = {3*e2_sum:.3f}')
    print(f'  G3: w3 * Σe3  = 2 × {e3_sum:.3f} = {2*e3_sum:.3f}')
    print(f'  G4: w4 * Σd   = 1 × {dp_sum+dm_sum:.3f} = {1*(dp_sum+dm_sum):.3f}')
    print(f'  {"─"*45}')
    print(f'  Weighted Z = {m2.ObjVal:.4f}')
    print(f'  Unweighted sum (raw deviations) = {cw_sum+e2_sum+e3_sum+dp_sum+dm_sum:.3f}')

    # ── Comparison with equal-weight run ─────────────────────────────────────
    print('\n' + '='*55)
    print('COMPARISON: Equal weights vs Weighted')
    print('='*55)
    print(f'  {"Goal":<30}  {"Equal (w=1)":>12}  {"Weighted":>10}')
    print('-'*55)
    print(f'  {"G1 Σcw (consec weekends)":<30}  {0:>12.3f}  {cw_sum:>10.3f}')
    print(f'  {"G2 Σe2 (nights 2 days apart)":<30}  {0:>12.3f}  {e2_sum:>10.3f}')
    print(f'  {"G3 Σe3 (night excess/week)":<30}  {0:>12.3f}  {e3_sum:>10.3f}')
    print(f'  {"G4 Σd (day/night imbalance)":<30}  {4:>12.3f}  {dp_sum+dm_sum:>10.3f}')
    print(f'  {"Z (objective value)":<30}  {4:>12.3f}  {m2.ObjVal:>10.3f}')

else:
    print(f'No optimal solution. Status: {m2.Status}')

Set parameter OutputFlag to value 1
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.3.0 25D125)

CPU model: Apple M1 Pro
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1946 rows, 936 columns and 5560 nonzeros (Min)
Model fingerprint: 0x4b25f0e8
Model has 312 linear objective coefficients
Variable types: 312 continuous, 624 integer (624 binary)
Coefficient statistics:
  Matrix range     [1e+00, 5e+00]
  Objective range  [1e+00, 4e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+00]

Presolve removed 1448 rows and 328 columns
Presolve time: 0.01s
Presolved: 498 rows, 608 columns, 2754 nonzeros
Variable types: 0 continuous, 608 integer (556 binary)
Found heuristic solution: objective 36.0000000

Root relaxation: objective 4.000000e+00, 231 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Inc

---
### 9.1 Weighted Goal Interpretation



I tried two different weight combinations — ($w_1=4,\ w_2=3,\ w_3=2,\ w_4=1$) and ($w_1=4,\ w_2=4,\ w_3=1,\ w_4=1$) — and both produced the **exact same schedule and POW assignments** as the equal-weight run, with $Z = 4$ in all cases.

---

#### Equal vs Weighted Goal

The reason there's no change is because G1, G2, and G3 were already at zero in the equal-weight solution, so there was nothing for the higher weights to penalise. Multiplying zero by 4 is still zero. The weights only matter when violations actually exist — since they don't here for G1–G3, changing those weights had no effect on the schedule.

That said, the weights are not useless. They define the **trade-off rules the solver would follow if a conflict came up**. The table below shows what each weight means in practice:

| Goal | Raw deviation | Equal-weight | Weighted | What the weight means to the solver |
|------|:---:|:---:|:---:|-----|
| G1: Consecutive weekends | 0 | $1 \times 0 = 0$ | $4 \times 0 = 0$ | Would accept up to **4 extra G4 deviations** before allowing even one consecutive-weekend violation |
| G2: Nights two days apart | 0 | $1 \times 0 = 0$ | $3 \times 0 = 0$ | Would accept **3 extra G4 deviations** before allowing a near-consecutive-night pairing |
| G3: Night excess per week | 0 | $1 \times 0 = 0$ | $2 \times 0 = 0$ | Would accept **2 extra G4 deviations** before allowing an extra night shift in a week |
| G4: Day/night imbalance | 4 | $1 \times 4 = 4$ | $1 \times 4 = 4$ | Treated as minor — accepted where mathematically unavoidable |
| **Total Z** | | **4** | **4** | |

The G4 imbalance (4 peds with one extra day shift) will not go away regardless of the weights, because it comes from the math itself — peds with an odd number of required shifts ($L_j = 3$ or $L_j = 5$) can never split their workload perfectly between day and night.

---

#### Final Summary for Meg

The fact that both weight settings give the same schedule is actually a good sign. It means the schedule is not just optimal for one specific set of priorities, it holds up across different priority orderings. Meg can share this schedule with the team knowing:

1. **All hospital policies are met** : where every hard constraint is satisfied.
2. **G1, G2, and G3 are fully satisfied** : no consecutive weekends, no near-consecutive nights, and no week with more than 2 nights for any ped.
3. **The only remaining deviation (G4) cannot be avoided** given the odd workload sizes, and it only affects four peds by one shift each.
4. **The schedule is stable** : tweaking the weights in future cycles is unlikely to produce a significant different result unless the underlying data changes (e.g., more availability restrictions or heavier workloads that force G1–G3 trade-offs).

---